## Imports

In [1]:
import os
import pathlib
import polars as pl

## Configs

In [2]:
PROJECT_ROOT = pathlib.Path(os.path.dirname(os.path.abspath(".")))

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
MOVIES_PATH = RAW_DATA_PATH / "movies.csv"
RATINGS_PATH = RAW_DATA_PATH / "ratings.csv"

PREPROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "preprocessed"
TRAIN_PATH = PREPROCESSED_DATA_PATH / "train.parquet"
TEST_PATH = PREPROCESSED_DATA_PATH / "test.parquet"

## EDA

In [3]:
movies = pl.read_csv(MOVIES_PATH)
movies.head()

movieId,title,genres
i64,str,str
1,"""Toy Story (1995)""","""Adventure|Animation|Children|C…"
2,"""Jumanji (1995)""","""Adventure|Children|Fantasy"""
3,"""Grumpier Old Men (1995)""","""Comedy|Romance"""
4,"""Waiting to Exhale (1995)""","""Comedy|Drama|Romance"""
5,"""Father of the Bride Part II (1…","""Comedy"""


In [4]:
ratings = pl.read_csv(RATINGS_PATH)
ratings.head()

userId,movieId,rating,timestamp
i64,i64,f64,i64
1,16,4.0,1217897793
1,24,1.5,1217895807
1,32,4.0,1217896246
1,47,4.0,1217896556
1,50,4.0,1217896523


In [5]:
df = movies.join(ratings, on="movieId")
df.head()

movieId,title,genres,userId,rating,timestamp
i64,str,str,i64,f64,i64
16,"""Casino (1995)""","""Crime|Drama""",1,4.0,1217897793
24,"""Powder (1995)""","""Drama|Sci-Fi""",1,1.5,1217895807
32,"""Twelve Monkeys (a.k.a. 12 Monk…","""Mystery|Sci-Fi|Thriller""",1,4.0,1217896246
47,"""Seven (a.k.a. Se7en) (1995)""","""Mystery|Thriller""",1,4.0,1217896556
50,"""Usual Suspects, The (1995)""","""Crime|Mystery|Thriller""",1,4.0,1217896523


In [6]:
df.describe()

statistic,movieId,title,genres,userId,rating,timestamp
str,f64,str,str,f64,f64,f64
"""count""",105339.0,"""105339""","""105339""",105339.0,105339.0,105339.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0,0.0
"""mean""",13381.312477,null,null,364.924539,3.51685,1.1304e9
"""std""",26170.456869,null,null,197.486905,1.044872,1.8027e8
"""min""",1.0,"""'71 (2014)""","""(no genres listed)""",1.0,0.5,8.28564954e8
"""25%""",1073.0,null,null,192.0,3.0,9.71100838e8
"""50%""",2497.0,null,null,383.0,3.5,1.1152e9
"""75%""",5991.0,null,null,557.0,4.0,1.2755e9
"""max""",149532.0,"""À nous la liberté (Freedom for…","""Western""",668.0,5.0,1.4524e9


In [7]:
df = df.filter(pl.col("title").is_not_null())

df = df.with_columns(pl.col("genres").fill_null("(no genres listed)"))

df = df.unique()

In [8]:
df.describe()

statistic,movieId,title,genres,userId,rating,timestamp
str,f64,str,str,f64,f64,f64
"""count""",105339.0,"""105339""","""105339""",105339.0,105339.0,105339.0
"""null_count""",0.0,"""0""","""0""",0.0,0.0,0.0
"""mean""",13381.312477,null,null,364.924539,3.51685,1.1304e9
"""std""",26170.456869,null,null,197.486905,1.044872,1.8027e8
"""min""",1.0,"""'71 (2014)""","""(no genres listed)""",1.0,0.5,8.28564954e8
"""25%""",1073.0,null,null,192.0,3.0,9.71100838e8
"""50%""",2497.0,null,null,383.0,3.5,1.1152e9
"""75%""",5991.0,null,null,557.0,4.0,1.2755e9
"""max""",149532.0,"""À nous la liberté (Freedom for…","""Western""",668.0,5.0,1.4524e9


## Train/Test Split

Per-user split by `timestamp`: for each user, the last `TEST_FRAC` share of their ratings (by time) goes to test, the rest to train. This avoids leaking future interactions into training and keeps every user represented in train.

In [9]:
TEST_FRAC = 0.2

df = df.sort(["userId", "timestamp"])

df = df.with_columns(
    pl.col("timestamp").rank("ordinal").over("userId").alias("_rank"),
    pl.len().over("userId").alias("_n_user"),
)

df = df.with_columns(
    (pl.col("_rank") > (pl.col("_n_user") * (1 - TEST_FRAC))).alias("_is_test")
)

train_df = df.filter(~pl.col("_is_test")).drop(["_rank", "_n_user", "_is_test"])
test_df = df.filter(pl.col("_is_test")).drop(["_rank", "_n_user", "_is_test"])

print(f"train: {train_df.shape}, test: {test_df.shape}")

train: (84013, 6), test: (21326, 6)


Drop cold-start rows from test: users or movies that don't appear in train can't be scored by a model trained only on train (no embedding/history for them), so they'd just distort the eval metrics.

In [10]:
train_users = set(train_df["userId"].unique())
train_movies = set(train_df["movieId"].unique())

test_df = test_df.filter(
    pl.col("userId").is_in(train_users) & pl.col("movieId").is_in(train_movies)
)

print(f"train: {train_df.shape}, test after removing cold-start rows: {test_df.shape}")

train: (84013, 6), test after removing cold-start rows: (19608, 6)


In [11]:
train_df

movieId,title,genres,userId,rating,timestamp
i64,str,str,i64,f64,i64
3421,"""Animal House (1978)""","""Comedy""",1,4.5,1217895728
2407,"""Cocoon (1985)""","""Comedy|Sci-Fi""",1,2.5,1217895736
2947,"""Goldfinger (1964)""","""Action|Adventure|Thriller""",1,3.5,1217895742
261,"""Little Women (1994)""","""Drama""",1,1.5,1217895750
2455,"""Fly, The (1986)""","""Drama|Horror|Sci-Fi|Thriller""",1,2.0,1217895762
…,…,…,…,…,…
32291,"""Melinda and Melinda (2004)""","""Comedy|Drama""",668,2.0,1328311614
85025,"""Eagle, The (2011)""","""Adventure|Drama""",668,2.5,1328311624
86835,"""Priest (2011)""","""Action|Horror|Sci-Fi|Thriller""",668,3.5,1328370185


In [12]:
test_df

movieId,title,genres,userId,rating,timestamp
i64,str,str,i64,f64,i64
912,"""Casablanca (1942)""","""Drama|Romance""",1,5.0,1217897623
1222,"""Full Metal Jacket (1987)""","""Drama|War""",1,5.0,1217897625
1136,"""Monty Python and the Holy Grai…","""Adventure|Comedy|Fantasy""",1,5.0,1217897630
48780,"""Prestige, The (2006)""","""Drama|Mystery|Sci-Fi|Thriller""",1,4.0,1217897632
1233,"""Boot, Das (Boat, The) (1981)""","""Action|Drama|War""",1,4.5,1217897648
…,…,…,…,…,…
122886,"""Star Wars: Episode VII - The F…","""Action|Adventure|Fantasy|Sci-F…",668,4.0,1451535779
131656,"""Shaun the Sheep Movie (2015)""","""Adventure|Animation|Children""",668,3.0,1451535823
51084,"""Music and Lyrics (2007)""","""Comedy|Romance""",668,2.5,1451675850


## Saving parquets

In [13]:
PREPROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

train_df.write_parquet(TRAIN_PATH)
test_df.write_parquet(TEST_PATH)

print(f"saved train to {TRAIN_PATH}, test to {TEST_PATH}")

saved train to /Users/onebanane/Projects/for_work/triton/data/preprocessed/train.parquet, test to /Users/onebanane/Projects/for_work/triton/data/preprocessed/test.parquet
